In [4]:
!pip install -q "PuLP==3.3.0" pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 81.4 MB/s eta 0:00:00


In [1]:
import itertools
import time
import platform
import pandas as pd
import pulp
assert pulp.__version__.split(".")[0] == "3", "Restart the Colab session, then select Runtime > Run all."

# (name, maximum roll length, required item sizes, item demands)
instances = [
    ("I01", 9,  [2, 3, 4],     [2, 2, 2]),
    ("I02", 10, [6],           [5]),
    ("I03", 10, [5],           [3]),
    ("I04", 12, [3, 5, 7],     [3, 2, 2]),
    ("I05", 15, [4, 6, 9],     [4, 3, 2]),
    ("I06", 20, [4, 7, 11],    [5, 4, 3]),
    ("I07", 20, [5, 8, 9],     [6, 4, 3]),
    ("I08", 25, [4, 6, 9, 13], [5, 6, 4, 3]),
    ("I09", 30, [6, 10, 14],   [5, 4, 3]),
    ("I10", 35, [4, 7, 11, 16],[5, 4, 3, 2]),
]

TIME_LIMIT = 5  # seconds per integer model, same for all three


In [2]:
def kantorovich(W, sizes, demands, relax=False):
    m = len(sizes)
    K = sum(demands)  # a simple upper bound: one item per roll
    category = "Continuous" if relax else "Integer"
    model = pulp.LpProblem("Kantorovich", pulp.LpMinimize)

    y = [pulp.LpVariable(f"y_{k}", 0, 1,
                         cat="Continuous" if relax else "Binary") for k in range(K)]
    x = {(i, k): pulp.LpVariable(f"x_{i}_{k}", lowBound=0, cat=category)
         for i in range(m) for k in range(K)}

    model += pulp.lpSum(y)
    for i in range(m):
        model += pulp.lpSum(x[i, k] for k in range(K)) >= demands[i]
    for k in range(K):
        model += pulp.lpSum(sizes[i] * x[i, k] for i in range(m)) <= W * y[k]
    return model


def gilmore_gomory(W, sizes, demands, relax=False):
    m = len(sizes)
    category = "Continuous" if relax else "Integer"
    model = pulp.LpProblem("Gilmore_Gomory", pulp.LpMinimize)

    # Enumerate all feasible non-empty patterns for these small instances.
    options = [range(W // size + 1) for size in sizes]
    patterns = [p for p in itertools.product(*options)
                if any(p) and sum(sizes[i] * p[i] for i in range(m)) <= W]
    x = [pulp.LpVariable(f"pattern_{j}", lowBound=0, cat=category)
         for j in range(len(patterns))]

    model += pulp.lpSum(x)
    for i in range(m):
        model += pulp.lpSum(patterns[j][i] * x[j] for j in range(len(patterns))) >= demands[i]
    return model


def arcflow(W, sizes, demands, relax=False):
    m = len(sizes)
    category = "Continuous" if relax else "Integer"
    model = pulp.LpProblem("Arcflow", pulp.LpMinimize)

    # Each arc: (start position, end position, item type).
    # The type -1 means unused capacity (loss arc).
    arcs = [(d, d + size, i) for i, size in enumerate(sizes)
            for d in range(W - size + 1)]
    arcs += [(d, d + 1, -1) for d in range(W)]

    x = [pulp.LpVariable(f"arc_{j}", lowBound=0, cat=category)
         for j in range(len(arcs))]
    z = pulp.LpVariable("rolls", lowBound=0, cat=category)
    model += z

    # Flow conservation: z paths go from node 0 to node W.
    for d in range(W + 1):
        incoming = pulp.lpSum(x[j] for j, (a, b, i) in enumerate(arcs) if b == d)
        outgoing = pulp.lpSum(x[j] for j, (a, b, i) in enumerate(arcs) if a == d)
        if d == 0:
            model += outgoing - incoming == z
        elif d == W:
            model += incoming - outgoing == z
        else:
            model += incoming == outgoing

    for i in range(m):
        model += pulp.lpSum(x[j] for j, (a, b, item) in enumerate(arcs)
                            if item == i) >= demands[i]
    return model


In [3]:
models = {
    "Kantorovich": kantorovich,
    "Gilmore-Gomory": gilmore_gomory,
    "Arcflow": arcflow,
}

rows = []
for name, W, sizes, demands in instances:
    for model_name, build in models.items():
        model = build(W, sizes, demands)
        solver = pulp.PULP_CBC_CMD(msg=False, timeLimit=TIME_LIMIT, threads=1)

        start = time.perf_counter()
        model.solve(solver)
        seconds = time.perf_counter() - start
        status = pulp.LpStatus[model.status]
        value = pulp.value(model.objective) if status == "Optimal" else None

        lp_model = build(W, sizes, demands, relax=True)
        lp_model.solve(pulp.PULP_CBC_CMD(msg=False, timeLimit=TIME_LIMIT, threads=1))
        lp_bound = (pulp.value(lp_model.objective)
                    if pulp.LpStatus[lp_model.status] == "Optimal" else None)

        rows.append({
            "Instance": name, "Model": model_name, "Status": status,
            "Rolls": value, "Time (s)": seconds,
            "Variables": len(model.variables()),
            "Constraints": len(model.constraints),
            "LP bound": lp_bound,
        })

results = pd.DataFrame(rows)
results["Gap"] = results["Rolls"] - results["LP bound"]
display(results.round(3))


,Instance,Model,Status,Rolls,Time (s),Variables,Constraints,LP bound,Gap
0,I01,Kantorovich,Optimal,2.0,0.014,24,9,2.000,0.000
1,I01,Gilmore-Gomory,Optimal,2.0,0.024,17,3,2.000,0.000
2,I01,Arcflow,Optimal,2.0,0.011,31,13,2.000,0.000
3,I02,Kantorovich,Optimal,5.0,0.008,10,6,3.000,2.000
4,I02,Gilmore-Gomory,Optimal,5.0,0.006,1,1,5.000,0.000
5,I02,Arcflow,Optimal,5.0,0.006,16,12,5.000,0.000
6,I03,Kantorovich,Optimal,2.0,0.007,6,4,1.500,0.500
7,I03,Gilmore-Gomory,Optimal,2.0,0.006,2,1,1.500,0.500
8,I03,Arcflow,Optimal,2.0,0.007,17,12,1.500,0.500
9,I04,Kantorovich,Optimal,3.0,0.009,28,10,2.750,0.250


In [4]:
summary = results.groupby("Model").agg(
    solved=("Status", lambda statuses: (statuses == "Optimal").sum()),
    mean_seconds=("Time (s)", "mean"),
    mean_variables=("Variables", "mean"),
    mean_constraints=("Constraints", "mean"),
    mean_gap=("Gap", "mean"),
).round(3)

display(summary)
results.to_csv("detailed_results.csv", index=False)
summary.to_csv("summary_results.csv")

# Capture simple hardware/software information for your report.
with open("/proc/cpuinfo") as f:
    cpu_line = next((s.strip() for s in f if s.startswith("model name")), "Unknown CPU")
with open("/proc/meminfo") as f:
    memory_line = next((s.strip() for s in f if s.startswith("MemTotal")), "Unknown RAM")
print("System:", platform.platform())
print("Processor:", cpu_line)
print("Memory:", memory_line)
print("Python:", platform.python_version())
print("PuLP:", pulp.__version__, "Solver: CBC")


,solved,mean_seconds,mean_variables,mean_constraints,mean_gap
Model,,,,,
Arcflow,10,0.012,60.4,22.4,0.340
Gilmore-Gomory,10,0.010,19.4,2.8,0.340
Kantorovich,10,0.011,41.2,12.7,0.658


System: Linux-6.6.122+-x86_64-with-glibc2.39
Processor: model name	: Intel(R) Xeon(R) CPU @ 2.20GHz
Memory: MemTotal:       13286944 kB
Python: 3.13.16
PuLP: 3.3.0 Solver: CBC
